In [ ]:
import os, shutil, gc, torch, warnings, random, time, json
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments,
    DataCollatorWithPadding, EarlyStoppingCallback,
    AutoModelForSequenceClassification, TrainerCallback  # ✅ 新增 TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# --- 环境变量 ---
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==================== 1. 全局配置 ====================
MODEL_NAME = "roberta-base"
NUM_LABELS = 5
MAX_LENGTH = 128
EPOCHS = 20
BATCH_SIZE = 64
RANDOM_SEEDS = [45, 123, 789, 2024, 1001]
FULL_LR = 2e-5
PEFT_LR = 3e-4

CONFIGS = {
    2300: {
        "train": {4: 1200, 3: 600, 2: 300, 1: 150, 0: 50},
        "eval_steps": 10, "memory_size": 1200, "temperature": 0.5, "loss_weight": 0.2,
        "warmup_steps": 30, "tail_weight": 1.0, "lr_scale": 0.9, "grad_acc": 1,
        "fusion_init": -1.8, "smoothing": 0.1, "clamp_weights": True
    },
    1150: {
        "train": {4: 600, 3: 300, 2: 150, 1: 80, 0: 20},
        "eval_steps": 10, "memory_size": 1200, "temperature": 0.15, "loss_weight": 0.1,
        "warmup_steps": 20, "tail_weight": 2.0, "lr_scale": 1.0, "grad_acc": 2,
        "fusion_init": 0.3, "smoothing": 0.05, "clamp_weights": True
    }
}
TAIL_CLASSES = [0, 1]

EXPERIMENTS = [
    {"name": "LoRA-Ours", "method": "peft", "loss_type": "original",
     "use_class_weight": True, "peft_type": "lora", "hsp": True, "memory_bank": True},
]

SENS_MEM_SIZES    = [200, 600, 1200, 2000]
SENS_TAIL_WEIGHTS = [1.0, 2.0, 3.0, 4.0]

MAIN_RESULTS_FILE = "sst5_ours_final.csv"
SENSITIVITY_FILE  = "sst5_sensitivity_add.csv"
GATE_LOG_DIR      = "./gate_logs_sst5"   # ✅ 门控过程日志目录
IMG_DATA_DIR      = "./viz_final_sst5"
os.makedirs(IMG_DATA_DIR, exist_ok=True)
os.makedirs(GATE_LOG_DIR, exist_ok=True)

# ==================== 2. Gate Monitor Callback ====================
class GateMonitorCallback(TrainerCallback):
    """每次 evaluate 时记录一次 σ(fusion_weight) 的当前值"""
    def __init__(self, model, log_path):
        self.model    = model
        self.log_path = log_path
        self.records  = []

    def on_evaluate(self, args, state, control, **kwargs):
        gate_val = torch.sigmoid(self.model.fusion_weight).item()
        self.records.append({
            "step":        state.global_step,
            "epoch":       round(state.epoch, 3) if state.epoch else 0,
            "gate_sigmoid": gate_val
        })

    def on_train_end(self, args, state, control, **kwargs):
        pd.DataFrame(self.records).to_csv(self.log_path, index=False)
        print(f"  [GateMonitor] Saved {len(self.records)} records → {self.log_path}")

# ==================== 3. 模型核心定义 ====================

def get_parameter_names(model, forbidden_layer_types):
    result = []
    for name, child in model.named_children():
        result += [f"{name}.{n}" for n in get_parameter_names(child, forbidden_layer_types)
                   if not isinstance(child, tuple(forbidden_layer_types))]
    result += list(model._parameters.keys())
    return result


class MemoryBank(nn.Module):
    def __init__(self, feature_dim=128, num_classes=5, memory_size=600, temperature=0.3,
                 tail_classes=[0, 1], tail_weight=1.3, warmup_steps=10, min_samples=5):
        super().__init__()
        self.feature_dim = feature_dim; self.num_classes = num_classes
        self.temperature = temperature; self.tail_classes = tail_classes
        self.tail_weight = tail_weight; self.warmup_steps = warmup_steps
        self.min_samples = min_samples; self.current_step = 0
        capacity = memory_size // num_classes
        for c in range(num_classes):
            self.register_buffer(f'memory_bank_{c}', torch.randn(capacity, feature_dim))
        self.register_buffer('bank_ptrs', torch.zeros(num_classes, dtype=torch.long))
        self.register_buffer('bank_sizes', torch.zeros(num_classes, dtype=torch.long))

    def get_memory_bank(self, class_id): return getattr(self, f'memory_bank_{class_id}')
    def set_memory_bank(self, class_id, data, start_idx, end_idx):
        getattr(self, f'memory_bank_{class_id}')[start_idx:end_idx] = data

    @torch.no_grad()
    def update_memory_bank(self, features, labels):
        if self.current_step < self.warmup_steps: return
        features = F.normalize(features.detach().clone(), dim=1)
        labels = labels.detach().clone()
        for c in range(self.num_classes):
            mask = (labels == c)
            if not mask.any(): continue
            feats_c = features[mask].clone(); n = feats_c.size(0)
            bank = self.get_memory_bank(c); ptr = self.bank_ptrs[c].item(); cap = bank.size(0)
            if ptr + n <= cap:
                self.set_memory_bank(c, feats_c, ptr, ptr + n)
                self.bank_ptrs[c] = (ptr + n) % cap
            else:
                rem = cap - ptr
                self.set_memory_bank(c, feats_c[:rem], ptr, cap)
                self.set_memory_bank(c, feats_c[rem:], 0, n - rem)
                self.bank_ptrs[c] = n - rem
            self.bank_sizes[c] = min(self.bank_sizes[c] + n, cap)

    def forward(self, features, labels):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            return torch.tensor(0.0, device=features.device, requires_grad=True)
        features_norm = F.normalize(features, dim=1)
        total_loss = 0.0; valid = 0
        for i in range(features.size(0)):
            feat = features_norm[i]; label = labels[i].item()
            pos = self.get_memory_bank(label)[:self.bank_sizes[label]].detach().clone()
            if pos.size(0) < self.min_samples: continue
            negs = [self.get_memory_bank(c)[:self.bank_sizes[c]].detach().clone()
                    for c in range(self.num_classes)
                    if c != label and self.bank_sizes[c] >= self.min_samples]
            if not negs: continue
            neg_feats = torch.cat(negs, dim=0)
            logits = torch.cat([
                torch.matmul(feat.unsqueeze(0), pos.t()) / self.temperature,
                torch.matmul(feat.unsqueeze(0), neg_feats.t()) / self.temperature
            ], dim=1)
            total_loss += (self.tail_weight if label in self.tail_classes else 1.0) * \
                          F.cross_entropy(logits, torch.zeros(1, dtype=torch.long, device=features.device))
            valid += 1
        return total_loss / valid if valid > 0 else torch.tensor(0.0, device=features.device, requires_grad=True)


class HierarchicalSmartPooling(nn.Module):
    def __init__(self, hs, dr=0.1):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(hs, hs), nn.Tanh(), nn.Linear(hs, 1), nn.Softmax(dim=1))
        self.fusion = nn.Sequential(nn.Linear(hs * 3, hs * 2), nn.LayerNorm(hs * 2),
                                    nn.GELU(), nn.Dropout(dr), nn.Linear(hs * 2, hs))

    def forward(self, x, m):
        w = self.attn(x).masked_fill(m.unsqueeze(-1) == 0, -1e9)
        w = F.softmax(w, dim=1)
        return self.fusion(torch.cat([
            torch.sum(x * w, 1),
            torch.sum(x * m.unsqueeze(-1), 1) / m.sum(1, keepdim=True).clamp(min=1e-9),
            x.masked_fill(m.unsqueeze(-1) == 0, -1e9).max(1)[0]
        ], dim=1))


class UnifiedModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        peft_type = cfg.get("peft_type", "lora")
        use_dora = True if peft_type == "dora" else False
        peft_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION, r=16, lora_alpha=32,
            lora_dropout=0.1, target_modules=["query", "key", "value"],
            use_dora=use_dora
        )
        self.encoder = get_peft_model(AutoModel.from_pretrained(MODEL_NAME), peft_config)
        self.config = self.encoder.config
        self.config.num_labels = NUM_LABELS
        hs = self.encoder.config.hidden_size
        self.classifier_base = nn.Linear(hs, NUM_LABELS)
        self.hsp_module = HierarchicalSmartPooling(hs)
        self.classifier_hsp = nn.Linear(hs, NUM_LABELS)
        nn.init.constant_(self.classifier_hsp.weight, 0)
        nn.init.constant_(self.classifier_hsp.bias, 0)
        self.fusion_weight = nn.Parameter(torch.tensor([cfg.get("fusion_init", 0.1)]))
        self.projector = nn.Sequential(
            nn.Linear(hs, hs), nn.ReLU(), nn.Dropout(0.1), nn.Linear(hs, 128)
        )

    def forward(self, input_ids, attention_mask, labels=None):
        hidden = self.encoder(input_ids, attention_mask).last_hidden_state
        cls_feat = hidden[:, 0, :]
        logits = self.classifier_base(cls_feat)
        logits = logits + torch.sigmoid(self.fusion_weight) * \
                 self.classifier_hsp(self.hsp_module(hidden, attention_mask))
        return {"loss": None, "logits": logits, "proj_features": self.projector(cls_feat)}


class UnifiedTrainer(Trainer):
    def __init__(self, loss_type, class_weights, cls_num_list, memory_loss,
                 loss_weight, is_peft, smoothing, use_class_weight=True, **kwargs):
        super().__init__(**kwargs)
        self.loss_type = loss_type
        self.use_class_weight = use_class_weight
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32) if class_weights is not None else None
        self.cls_num_list = cls_num_list
        self.memory_loss_module = memory_loss
        self.aux_loss_weight = loss_weight
        self.is_peft = is_peft
        self.label_smoothing = smoothing
        self.current_epoch = 0

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(inputs["input_ids"], inputs["attention_mask"], labels)
        logits = outputs["logits"]
        weight_to_use = None
        if self.use_class_weight and self.class_weights is not None:
            if self.class_weights.device != logits.device:
                self.class_weights = self.class_weights.to(logits.device)
            weight_to_use = self.class_weights
        loss_fct = nn.CrossEntropyLoss(weight=weight_to_use, label_smoothing=self.label_smoothing)
        total_loss = loss_fct(logits.view(-1, NUM_LABELS), labels.view(-1))
        if self.is_peft and self.memory_loss_module is not None and outputs.get("proj_features") is not None:
            proj_features = outputs["proj_features"]
            loss_mb = self.memory_loss_module(proj_features, labels)
            total_loss += self.aux_loss_weight * loss_mb
            with torch.no_grad():
                self.memory_loss_module.update_memory_bank(proj_features, labels)
        return (total_loss, SequenceClassifierOutput(loss=total_loss, logits=logits)) if return_outputs else total_loss

    def create_optimizer(self):
        if self.optimizer is None:
            decay_parameters = get_parameter_names(self.model, [nn.LayerNorm])
            decay_parameters = [name for name in decay_parameters if "bias" not in name]
            optimizer_grouped_parameters = []
            for n, p in self.model.named_parameters():
                if not p.requires_grad: continue
                if "fusion_weight" in n:
                    optimizer_grouped_parameters.append({"params": [p], "weight_decay": 0.0,
                                                          "lr": self.args.learning_rate * 5})
                else:
                    optimizer_grouped_parameters.append({
                        "params": [p],
                        "weight_decay": self.args.weight_decay if n in decay_parameters else 0.0,
                        "lr": self.args.learning_rate
                    })
            optimizer_cls, optimizer_kwargs = Trainer.get_optimizer_cls_and_kwargs(self.args)
            self.optimizer = optimizer_cls(optimizer_grouped_parameters, **optimizer_kwargs)
        return self.optimizer

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.current_epoch = state.epoch


# ==================== 4. 辅助函数 ====================
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def get_custom_dataset(df, config, seed):
    sampled = [df[df['label'] == l].sample(n=min(len(df[df['label'] == l]), c), random_state=seed)
               for l, c in config.items()]
    return pd.concat(sampled).sample(frac=1, random_state=seed).reset_index(drop=True)


def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    preds = np.argmax(logits, axis=-1)
    labels = eval_pred.label_ids
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    recalls = [report[str(i)]['recall'] for i in range(NUM_LABELS)]
    try:
        probs = F.softmax(torch.tensor(logits), dim=-1).numpy()
        auc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except:
        auc = 0.0
    metrics = {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
        "accuracy": accuracy_score(labels, preds),
        "balanced_acc": np.mean(recalls),
        "g_mean": np.prod(recalls) ** (1 / NUM_LABELS),
        "auc": auc
    }
    for i in range(NUM_LABELS):
        metrics[f"f1_class_{i}"] = report[str(i)]['f1-score']
    return metrics


def append_to_csv(filename, row_dict):
    df = pd.DataFrame([row_dict])
    df.to_csv(filename, mode='a', header=not os.path.exists(filename), index=False)


# ==================== 5. 数据加载 ====================
print(">>> Loading SST-5 Dataset...")
dataset_raw = load_dataset("SetFit/sst5")
full_df = pd.DataFrame(dataset_raw["train"]).dropna(subset=["text", "label"])
full_df["label"] = full_df["label"].astype(int)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(ex):
    return tokenizer(ex["text"], truncation=True, max_length=MAX_LENGTH)


train_pool, val_pool = train_test_split(full_df, test_size=0.2, stratify=full_df["label"], random_state=42)
val_df = get_custom_dataset(val_pool, {k: 80 for k in range(NUM_LABELS)}, 42)
val_ds = Dataset.from_pandas(val_df).map(tokenize, batched=True).select_columns(
    ["input_ids", "attention_mask", "label"]
)

if os.path.exists(MAIN_RESULTS_FILE): os.remove(MAIN_RESULTS_FILE)
if os.path.exists(SENSITIVITY_FILE):  os.remove(SENSITIVITY_FILE)

# ==================== PART A: 主实验 ====================
print(f"\n{'=' * 80}\nPART A: MAIN EXPERIMENT (SST-5)\n{'=' * 80}")
for N in [2300, 1150]:
    cfg = CONFIGS[N]
    for exp in EXPERIMENTS:
        safe_name = exp['name'].replace('/', '_').replace('+', '_plus')
        for seed in RANDOM_SEEDS:
            print(f"\n[Run] N={N} | {exp['name']} | Seed={seed}")
            set_seed(seed)
            train_df = get_custom_dataset(train_pool, cfg["train"], seed)
            cw = compute_class_weight('balanced', classes=np.unique(train_df['label']), y=train_df['label'])
            class_weights_np = torch.tensor(cw, dtype=torch.float).clamp(max=10.0).numpy() \
                if cfg['clamp_weights'] else cw
            cls_num_list = [len(train_df[train_df['label'] == i]) for i in range(NUM_LABELS)]
            train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True).select_columns(
                ["input_ids", "attention_mask", "label"]
            )
            model_cfg = exp.copy()
            model_cfg["fusion_init"] = cfg["fusion_init"]
            model = UnifiedModel(model_cfg).to(device)
            output_dir_path = f"./tmp_sst5_{N}_{safe_name}_{seed}"

            # ✅ 门控过程日志
            gate_log_path = f"{GATE_LOG_DIR}/gate_N{N}_{safe_name}_seed{seed}.csv"
            gate_cb = GateMonitorCallback(model, gate_log_path)

            trainer = UnifiedTrainer(
                loss_type=exp["loss_type"],
                class_weights=class_weights_np,
                cls_num_list=cls_num_list,
                memory_loss=MemoryBank(
                    128, NUM_LABELS, cfg["memory_size"], cfg["temperature"],
                    TAIL_CLASSES, cfg["tail_weight"], cfg["warmup_steps"], 5
                ).to(device),
                loss_weight=cfg["loss_weight"],
                is_peft=True,
                smoothing=cfg["smoothing"],
                use_class_weight=exp.get("use_class_weight", True),
                model=model,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                tokenizer=tokenizer,
                data_collator=DataCollatorWithPadding(tokenizer),
                compute_metrics=compute_metrics,
                args=TrainingArguments(
                    output_dir=output_dir_path,
                    num_train_epochs=EPOCHS,
                    per_device_train_batch_size=BATCH_SIZE,
                    gradient_accumulation_steps=cfg["grad_acc"],
                    learning_rate=PEFT_LR * cfg["lr_scale"],
                    warmup_ratio=0.1,
                    weight_decay=0.01,
                    eval_strategy="steps",
                    eval_steps=cfg["eval_steps"],
                    save_strategy="steps",
                    save_steps=cfg["eval_steps"],
                    save_total_limit=1,
                    load_best_model_at_end=True,
                    metric_for_best_model="macro_f1",
                    fp16=True,
                    report_to="none",
                    logging_steps=5
                ),
                # ✅ 加入门控监控 callback
                callbacks=[EarlyStoppingCallback(early_stopping_patience=8), gate_cb]
            )

            torch.cuda.reset_peak_memory_stats()
            start_time = time.time()
            trainer.train()
            train_runtime = time.time() - start_time
            peak_memory = torch.cuda.max_memory_allocated() / 1024 / 1024
            res = trainer.evaluate()
            start_inf = time.time(); _ = trainer.predict(val_ds); inf_time = time.time() - start_inf

            gate_val = torch.sigmoid(model.fusion_weight).item()
            num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            row = {
                "Dataset": "SST-5", "N": N, "Method": exp['name'], "Seed": seed,
                "Macro-F1": res['eval_macro_f1'], "Weighted-F1": res['eval_weighted_f1'],
                "Accuracy": res['eval_accuracy'], "Balanced_Acc": res['eval_balanced_acc'],
                "G-Mean": res['eval_g_mean'], "AUC": res['eval_auc'],
                "Gate_Final_σ(λ)": gate_val,
                "Train_Time_Sec": train_runtime, "Inference_Time_Sec": inf_time,
                "Peak_Memory_MB": peak_memory, "Params_M": num_params / 1e6
            }
            for i in range(NUM_LABELS):
                row[f"F1_Class_{i}"] = res[f"eval_f1_class_{i}"]
            append_to_csv(MAIN_RESULTS_FILE, row)
            del model, trainer
            torch.cuda.empty_cache(); gc.collect()
            shutil.rmtree(output_dir_path, ignore_errors=True)


# ==================== PART B: 敏感性分析 ====================
print(f"\n{'=' * 80}\nPART B: SENSITIVITY (MemSize & TailWeight)\n{'=' * 80}")
cfg_s = CONFIGS[2300]
exp_ours = EXPERIMENTS[0]


def run_sens_sst5(param_name, values):
    for val in values:
        for seed in RANDOM_SEEDS:
            print(f"\n[Sens] {param_name}={val} | Seed={seed}")
            set_seed(seed)
            train_df = get_custom_dataset(train_pool, cfg_s["train"], seed)
            cw = compute_class_weight('balanced', classes=np.unique(train_df['label']), y=train_df['label'])
            class_weights_np = torch.tensor(cw, dtype=torch.float).clamp(max=10.0).numpy()
            train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True).select_columns(
                ["input_ids", "attention_mask", "label"]
            )
            cur_mem  = val if param_name == "MemSize"    else cfg_s["memory_size"]
            cur_tw   = val if param_name == "TailWeight" else cfg_s["tail_weight"]

            m_cfg = exp_ours.copy()
            m_cfg["fusion_init"] = cfg_s["fusion_init"]
            model = UnifiedModel(m_cfg).to(device)

            # ✅ 敏感性分析也记录门控过程
            gate_log_path = f"{GATE_LOG_DIR}/gate_sens_{param_name}_{val}_seed{seed}.csv"
            gate_cb = GateMonitorCallback(model, gate_log_path)

            trainer = UnifiedTrainer(
                loss_type="original",
                class_weights=class_weights_np,
                cls_num_list=[len(train_df[train_df['label'] == i]) for i in range(NUM_LABELS)],
                memory_loss=MemoryBank(
                    128, NUM_LABELS, cur_mem, cfg_s["temperature"],
                    TAIL_CLASSES, cur_tw, cfg_s["warmup_steps"], 5
                ).to(device),
                loss_weight=cfg_s["loss_weight"],
                is_peft=True,
                smoothing=cfg_s["smoothing"],
                use_class_weight=True,
                model=model,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                tokenizer=tokenizer,
                data_collator=DataCollatorWithPadding(tokenizer),
                compute_metrics=compute_metrics,
                args=TrainingArguments(
                    output_dir=f"./tmp_sens_{param_name}_{val}_{seed}",
                    num_train_epochs=EPOCHS,
                    per_device_train_batch_size=BATCH_SIZE,
                    gradient_accumulation_steps=cfg_s["grad_acc"],
                    learning_rate=PEFT_LR * cfg_s["lr_scale"],
                    warmup_ratio=0.1,
                    weight_decay=0.01,
                    eval_strategy="steps",
                    eval_steps=cfg_s["eval_steps"],
                    save_strategy="steps",
                    save_steps=cfg_s["eval_steps"],
                    save_total_limit=1,
                    load_best_model_at_end=True,
                    metric_for_best_model="macro_f1",
                    fp16=True,
                    report_to="none",
                    logging_steps=5
                ),
                callbacks=[EarlyStoppingCallback(early_stopping_patience=8), gate_cb]  # ✅
            )
            trainer.train()
            res = trainer.evaluate()
            gate_val = torch.sigmoid(model.fusion_weight).item()
            append_to_csv(SENSITIVITY_FILE, {
                "Type": param_name, "Value": val, "Seed": seed,
                "Macro_F1":    res['eval_macro_f1'],
                "Weighted-F1": res['eval_weighted_f1'],
                "Accuracy":    res['eval_accuracy'],
                "G-Mean":      res['eval_g_mean'],
                "AUC":         res['eval_auc'],
                "Gate_Final_σ(λ)": gate_val
            })
            del model, trainer
            torch.cuda.empty_cache(); gc.collect()
            shutil.rmtree(f"./tmp_sens_{param_name}_{val}_{seed}", ignore_errors=True)


run_sens_sst5("MemSize",    SENS_MEM_SIZES)
run_sens_sst5("TailWeight", SENS_TAIL_WEIGHTS)

print(f"\n{'=' * 80}\nSST-5 DONE!\n{'=' * 80}")
print(f"  主实验结果:   {MAIN_RESULTS_FILE}")
print(f"  敏感性结果:   {SENSITIVITY_FILE}")
print(f"  门控过程日志: {GATE_LOG_DIR}/  (每个run一个CSV，列: step / epoch / gate_sigmoid)")

2026-03-01 10:51:34.737558: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772362294.930767      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772362294.983657      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Map:   0%|          | 0/2130 [00:00<?, ? examples/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Macro F1,Weighted F1,Accuracy,Balanced Acc,G Mean,Auc,F1 Class 0,F1 Class 1,F1 Class 2,F1 Class 3,F1 Class 4
10,1.886300,2.678416,0.066667,0.066667,0.200000,0.200000,0.000000,0.529469,0.333333,0.000000,0.000000,0.000000,0.000000
20,3.642500,4.545859,0.066667,0.066667,0.200000,0.200000,0.000000,0.533027,0.333333,0.000000,0.000000,0.000000,0.000000
30,3.696900,4.628920,0.066667,0.066667,0.200000,0.200000,0.000000,0.542461,0.333333,0.000000,0.000000,0.000000,0.000000
40,3.656300,4.602745,0.066667,0.066667,0.200000,0.200000,0.000000,0.563992,0.333333,0.000000,0.000000,0.000000,0.000000
50,3.815200,4.678509,0.066667,0.066667,0.200000,0.200000,0.000000,0.576641,0.333333,0.000000,0.000000,0.000000,0.000000
60,3.692800,4.609945,0.066667,0.066667,0.200000,0.200000,0.000000,0.627258,0.333333,0.000000,0.000000,0.000000,0.000000
70,3.610900,4.562261,0.066667,0.066667,0.200000,0.200000,0.000000,0.702688,0.333333,0.000000,0.000000,0.000000,0.000000
80,3.590500,4.465194,0.216401,0.216401,0.367500,0.367500,0.000000,0.750406,0.456456,0.000000,0.000000,0.000000,0.625551
90,3.493700,4.435370,0.224098,0.224098,0.380000,0.380000,0.000000,0.761195,0.494915,0.000000,0.000000,0.023256,0.602317
100,3.439000,4.436491,0.223996,0.223996,0.377500,0.377500,0.000000,0.758242,0.483444,0.000000,0.000000,0.022989,0.613546


  [GateMonitor] Saved 36 records → ./gate_logs_sst5/gate_sens_TailWeight_4.0_seed1001.csv



SST-5 DONE!
  主实验结果:   sst5_ours_final.csv
  敏感性结果:   sst5_sensitivity_full.csv
  门控过程日志: ./gate_logs_sst5/  (每个run一个CSV，列: step / epoch / gate_sigmoid)
